# Phase 3 — Random Forest Baseline + GBDT Training

**Steps 3.1 – 3.6** from the CartIQ proposal.

| Step | Description |
|---|---|
| 3.1 | Train Random Forest baseline (200 trees, 5-fold CV) |
| 3.2 | Evaluate RF on held-out test set |
| 3.3 | Train LightGBM with scale_pos_weight |
| 3.4 | Run Optuna hyperparameter search (50 trials) |
| 3.5 | Evaluate GBDT on test set |
| 3.6 | Generate SHAP summary and bar plots |

**Prerequisite:** Run `02_feature_engineering.ipynb` first.

In [3]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import optuna
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

FEATURES_DIR = '../features'
MODELS_DIR   = '../saved_models'
OUTPUTS_DIR  = '../outputs'

os.makedirs(MODELS_DIR, exist_ok=True)
sns.set_style('darkgrid')
optuna.logging.set_verbosity(optuna.logging.WARNING)
print('Setup complete.')

C:\Users\Yunus Kounkourou\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete.


In [4]:
feature_df = pd.read_parquet(f'{FEATURES_DIR}/feature_matrix.parquet')

FEATURE_COLS = [
    'total_orders', 'avg_basket_size', 'avg_days_between_orders', 'user_reorder_rate',
    'product_order_freq', 'product_reorder_rate', 'avg_add_to_cart_position',
    'up_times_ordered', 'up_times_reordered', 'up_reorder_ratio',
    'up_days_since_last', 'up_order_streak',
    'order_dow', 'order_hour_of_day', 'days_since_last_order', 'purchase_velocity',
]

X_train = feature_df[feature_df['split'] == 'train'][FEATURE_COLS]
y_train = feature_df[feature_df['split'] == 'train']['reordered']
X_val   = feature_df[feature_df['split'] == 'val'][FEATURE_COLS]
y_val   = feature_df[feature_df['split'] == 'val']['reordered']
X_test  = feature_df[feature_df['split'] == 'test'][FEATURE_COLS]
y_test  = feature_df[feature_df['split'] == 'test']['reordered']

print(f'Train: {X_train.shape}  |  Val: {X_val.shape}  |  Test: {X_test.shape}')

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight (neg/pos): {scale_pos_weight:.2f}')

Train: (11616502, 16)  |  Val: (848147, 16)  |  Test: (843304, 16)
scale_pos_weight (neg/pos): 16.56


## Step 3.1 — Random Forest Baseline (200 trees, 5-fold CV)

In [ ]:
print('Training Random Forest (200 trees)...')
rf = RandomForestClassifier(
    n_estimators=200,
    n_jobs=-1,
    random_state=42,
    class_weight='balanced',
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Sample 100k rows for CV speed (full dataset takes very long for RF)
sample_idx = np.random.RandomState(42).choice(len(X_train), size=min(100000, len(X_train)), replace=False)
X_cv = X_train.iloc[sample_idx]
y_cv = y_train.iloc[sample_idx]

cv_f1 = cross_val_score(rf, X_cv, y_cv, cv=cv, scoring='f1', n_jobs=-1)
print(f'5-fold CV F1 (on 100k sample): {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')

rf.fit(X_train, y_train)
print('RF training complete.')

Training Random Forest (200 trees)...
5-fold CV F1 (on 100k sample): 0.0170 ± 0.0037


## Step 3.2 — Evaluate Random Forest on Test Set

In [ ]:
rf_proba = rf.predict_proba(X_test)[:, 1]
rf_pred  = (rf_proba >= 0.5).astype(int)

rf_metrics = {
    'precision': precision_score(y_test, rf_pred, zero_division=0),
    'recall':    recall_score(y_test, rf_pred, zero_division=0),
    'f1':        f1_score(y_test, rf_pred, zero_division=0),
    'roc_auc':   roc_auc_score(y_test, rf_proba),
}

print('Random Forest — Test Set Metrics')
print('-' * 40)
for k, v in rf_metrics.items():
    print(f'  {k:<12}: {v:.4f}')

joblib.dump(rf, f'{MODELS_DIR}/random_forest.joblib')
print(f'Model saved to {MODELS_DIR}/random_forest.joblib')

## Step 3.3 & 3.4 — LightGBM + Optuna Hyperparameter Search (50 trials)

In [ ]:
def objective(trial):
    params = {
        'objective':         'binary',
        'metric':            'binary_logloss',
        'verbosity':         -1,
        'scale_pos_weight':  scale_pos_weight,
        'n_estimators':      500,
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 300),
        'max_depth':         trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }

    lgb_train_ds = lgb.Dataset(X_train, label=y_train)
    lgb_val_ds   = lgb.Dataset(X_val,   label=y_val, reference=lgb_train_ds)
    callbacks    = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]

    model = lgb.train(params, lgb_train_ds, valid_sets=[lgb_val_ds], callbacks=callbacks)

    val_proba = model.predict(X_val)
    return f1_score(y_val, val_proba >= 0.5, zero_division=0)


print('Running Optuna hyperparameter search (50 trials)...')
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'Best trial F1 (val): {study.best_value:.4f}')
print(f'Best params: {study.best_params}')

In [ ]:
# Optuna optimization history
fig = optuna.visualization.matplotlib.plot_optimization_history(study)
plt.title('Optuna Optimization History (F1 on Val)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUTPUTS_DIR}/03_optuna_history.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3.5 — Train Final GBDT & Evaluate on Test Set

In [ ]:
best_params = dict(study.best_params)
best_params.update({
    'objective':        'binary',
    'metric':           'binary_logloss',
    'verbosity':        -1,
    'scale_pos_weight': scale_pos_weight,
    'n_estimators':     1000,
})

lgb_train_ds = lgb.Dataset(X_train, label=y_train, feature_name=FEATURE_COLS)
lgb_val_ds   = lgb.Dataset(X_val,   label=y_val,   reference=lgb_train_ds)
callbacks    = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]

print('Training final LightGBM model...')
gbdt_model = lgb.train(
    best_params, lgb_train_ds,
    valid_sets=[lgb_val_ds],
    callbacks=callbacks,
)

gbdt_proba = gbdt_model.predict(X_test)
gbdt_pred  = (gbdt_proba >= 0.5).astype(int)

gbdt_metrics = {
    'precision': precision_score(y_test, gbdt_pred, zero_division=0),
    'recall':    recall_score(y_test, gbdt_pred, zero_division=0),
    'f1':        f1_score(y_test, gbdt_pred, zero_division=0),
    'roc_auc':   roc_auc_score(y_test, gbdt_proba),
}

print('LightGBM GBDT — Test Set Metrics')
print('-' * 40)
for k, v in gbdt_metrics.items():
    print(f'  {k:<12}: {v:.4f}')

gbdt_model.save_model(f'{MODELS_DIR}/lightgbm_model.txt')
print(f'Model saved to {MODELS_DIR}/lightgbm_model.txt')

## Step 3.6 — SHAP Feature Importance

In [ ]:
print('Computing SHAP values on 5,000 test samples...')
X_shap = X_test.sample(5000, random_state=42)

explainer   = shap.TreeExplainer(gbdt_model)
shap_values = explainer.shap_values(X_shap)

# Summary plot (beeswarm)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=FEATURE_COLS, show=False)
plt.title('GBDT Feature Importance — SHAP Summary', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUTPUTS_DIR}/03_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Bar plot (mean |SHAP|)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_shap, feature_names=FEATURE_COLS,
                  plot_type='bar', show=False)
plt.title('GBDT Feature Importance — SHAP Bar', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUTPUTS_DIR}/03_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Interim comparison table (RF vs GBDT)
results = {'random_forest': rf_metrics, 'gbdt': gbdt_metrics}

print(f'\n{"Model":<20} {"Precision":<12} {"Recall":<12} {"F1":<12} {"ROC-AUC":<12}')
print('-' * 68)
for name, m in results.items():
    print(f'{name:<20} {m["precision"]:<12.4f} {m["recall"]:<12.4f} {m["f1"]:<12.4f} {m["roc_auc"]:<12.4f}')

with open(f'{OUTPUTS_DIR}/model_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nResults saved to {OUTPUTS_DIR}/model_results.json')
print('Phase 3 complete. Proceed to 04_ncf_model.ipynb')